In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    coalesce,
    concat_ws,
    current_date,
    current_timestamp,
    datediff,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
    when,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_exposure_arotron(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        # "ExposureId string, SourceSystem string, PortfolioKey string, PortfolioName string,"
        # "IngestionTimestamp timestamp, BatchId string, ExposureDebtorKey string, CaseCode int,"
        # "ExposureSourceSystemKey string, AgreementKey string, PartyKey string, ExposureKey string,"
        # "PartyTaxRegistrationNumber string, CDI string, PartyName string, PartyRelationshipTypeCode string,"
        # "PartyRelationshipType string, PrimaryPartyFlag boolean, AllBorrowersDeceasedFlag boolean,"
        # "ProductKey string, ProductCode int, ExposureTypeCode string, ExposureType string,"
        # "ExposureStatusCode string, ExposureStatus string, ExposureStatusdate date, Opendate date,"
        # "ActiveExpiryDate date, Maturitydate date, OriginBank string, OriginationAmount decimal(19,2),"
        # "Currency string, BankingUnitCode int, BankingUnit string, MechanismCode string, Mechanism string,"
        # "BranchCode string, Branch string, PurposeReasonCode int, PurposeReason string,"
        # "PurposeKTCode int, PurposeKT string, MaterialityStartDate timestamp,"
        # "DaysPastDueWithMateriality decimal(15,0), DaysPastDue int, DaysPastDuePrincipal int,"
        # "DaysPastDueinterest string, BucketCode string, Bucket string, FirstArreardate date,"
        # "ExpectedOverduedate date, Overduedate date, Defaultdate date,  Denounceddate date,"
        # "MasterExposureKey string, SlaveFlag boolean,"
        # "InstalmentFrequency int, InstallmentFixedAmount decimal(19,2), NumberOfInstallments int,"
        # "FirstInstallmentDate date, NextInstallmentdate date, NextInstallmentAmount decimal(19,2),"
        # "PreviousInstallmentdate date, LastBillingdate date, NumberOfUnpaidBills string,"
        # "LastRestructuringTypeCode int, LastRestructuringType string, LastRestructuringDate date, RestructuringPEEdate date,"
        # "RestructuringPEETypeCode int, RestructuringPEEType string, ObjectiveStatusCode string,"
        # "ObjectiveStatus string, ObjectiveStatusdate date, GracePeriodEnddate date,"
        # "GracePeriodTypeCode string, GracePeriodType string, GracePeriodinterestPayableFlag string,"
        # "GracePeriodNextinterestdate timestamp, GracePeriodinterestFrequency decimal(3,0),"
        # "OffBalance decimal(19,2), OnBalance decimal(19,2), OutstandingTotalBalance string,"
        # "Principal decimal(19,2), interest decimal(19,2), Unrealizedinterest decimal(19,2),"
        # "Commission  decimal(19,2), Expense decimal(19,2), Accrued decimal(19,2),"
        # "DueAmount decimal(26,4), DuePrincipal decimal (19,2), Dueinterest decimal(19,2),"
        # "DueUnrealizedinterest decimal(19,2), DueCommission decimal (19,2), DueExpense decimal(19,2),"
        # "DueAccrued decimal(19,2), Penaltyinterest decimal(19,2),PenaltyUnrealizedinterest decimal(19,2),"
        # "InterestRate decimal(19,6), InterestCode int, InterestDescription string,"
        # "BaseRate decimal(19,6), BaseCode int, BaseDescription string,"
        # "N128Rate decimal(19,6), SpreadRate decimal(19,6), PenaltyRate decimal(19,6),"
        # "InterestRateTypeCode string, InterestRateType string, SpreadCurrencyCode string,"
        # "SpreadCurrency string, SpreadPeriod string, InterestDuration int, DayCounts int,"
        # "InterestFrequency string, InterestFixedDate date, interestValiditydate date,"
        # "InterestChangeStartdate date, InterestChangeEnddate date, InterestRateUpdatedate date,"
        # "InterestCalcStopFlag string, AutoRenewalFlag string, IsActiveFlag boolean,WriteOffAmount decimal(19,4),"
        # "WriteOffDate date,AssetSegmentCode string, AssetSegment string, DiasNumber string,"
        # "ProvisionFrozenInterestAmount string, AccountNticeBUKey string, SayFlag string,"
        # "InsuranceCompany string, PhysicalNumber string, PreCourtDate string, CourtDate string,"
        # "ContractSigningDate string, ExposureLimitAmount string, BaloonAmount decimal(15,2),"
        # "AdministrationStatusCode integer, AdministrationStatus integer, Tenor string,"
        # "IsSoldFlag boolean, IsSubsidyFlag string, IsGuaranteeFlag boolean, EverDenouncedFlag string,"
        # "EverDenouncedDate string, SPVCode string, SPV string, TotalOverdueInsterest decimal(4,0),"
        # "TotalOverdueInstallments decimal(4,0), OutstandingTotalBalanceMLT decimal(19,4),"
        # "TreatmentBranchCode int, TreatmentBranch string, LetterStatusCode string, LetterStatus string,"
        # "LetterStatusDate date, CoCStatusCode string, CoCStatus string, CoCAccountStatusDate date,"
        # "FirstLetterDate date, SecondLetterDate date, ThirdLetterDate date,"
        # "FirstLetterReminderDate date, FirstLetterFlag boolean, FirstLetterReminderFlag boolean,"
        # "SecondLetterFlag boolean, ThirdLetterFlag boolean, ExclusionDate date, ExclusionReasonCode string,"
        # "ExclusionReason string, LegalActionPriority string, DueDebtAmount decimal(19,4), BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.Exposure"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                    spark,
                    curated_table_path,
                    run_date,  # type: ignore
                "arotron",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter

        portfolio_selected_fields = ["PortfolioName", "Prefix", "unit_code_profits"]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )


        # ================== eaiSIG_BE ==================

        eaiSIG_BE_extra_filters = None

        if eaiSIG_BE_extra_filters is not None:
            eaiSIG_BE_extra_filters.append(date_filter)
            eaiSIG_BE_final_filters = " AND ".join(eaiSIG_BE_extra_filters)
        else:
            eaiSIG_BE_final_filters = date_filter

        eaiSIG_BE_selected_fields = [
            "eaiSIG_BE_CASE_S_ID",
            "LoadID",
            "eaiSIG_BE_CASE_CONTRNUM",
            "eaiSIG_BE_CUST_S_UNID",
            "eaiSIG_BE_SPRD_S_ID",
            "eaiSIG_BE_CASE_OPNDATE",
            "eaiSIG_BE_CASE_EXPDATE",
            "eaiSIG_BE_CASE_SPENDLIM",
            "eaiSIG_BE_CURR_CODE",
            "eaiSIG_BE_CASE_SERVICE_TYPE",
            "eaiSIG_BE_CASE_BRANCH",
            "eaiSIG_BE_DEL_DAYS",
            "eaiSIG_BE_CASE_COLFRSDATE",
            "eaiSIG_BE_CASE_STATUS1",
            "eaiSIG_BE_CASE_STATUS1_DATE",
            "eaiSIG_BE_CASE_MASTER_ACC",
            "eaiSIG_BE_CASE_PERIODCITY",
            "eaiSIG_BE_CASE_EMI_AMOUNT",
            "eaiSIG_BE_CASE_DURATION",
            "eaiSIG_BE_CASE_DUEDATE",
            "eaiSIG_BE_CASE_GRCP_EXPDATE",
            "eaiSIG_BE_CASE_DEBT",
            "eaiSIG_BE_CASE_OUTBAL",
            "eaiSIG_BE_CASE_CAPAMT",
            "eaiSIG_BE_CASE_REGINT2",
            "eaiSIG_BE_CASE_FRZAMT",
            "eaiSIG_BE_CASE_OTHERAMT1",
            "eaiSIG_BE_CASE_OVRDINT2",
            "eaiSIG_BE_CASE_OVRDINT1",
            "eaiSIG_BE_CASE_WOFFAMT_INTEREST",
            "eaiSIG_BE_CASE_INT_RATE",
            "eaiSIG_BE_CASE_INTEREST_RATE_CATEGORY",
            "eaiSIG_BE_INTEREST_RATE_BASE",
            "eaiSIG_BE_CASE_PRCT_CONTRB",
            "eaiSIG_BE_SPREAD_INTEREST_MARGIN",
            "eaiSIG_BE_CASE_INTERESTRATE_DEFAULT",
            "eaiSIG_BE_CASE_RATE_CAT",
            "eaiSIG_BE_CASE_INTERESTRATE_EXPDATE",
            "eaiSIG_BE_CASE_ISOPEN",
            "eaiSIG_BE_CASE_STATUS3_DATE",
            "eaiSIG_BE_CASE_SPV",
            "eaiSIG_BE_CASE_BUSUNIT",
            "eaiSIG_BE_CASE_STATUS4_DATE",
            "businessdate",
        ]

        eaiSIG_BE = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotron_eaiSIG_BE",
            eaiSIG_BE_final_filters,
            eaiSIG_BE_selected_fields,
        )

        # ================== eaiSIG_BPR ==================

        eaiSIG_BPR_extra_filters = None

        if eaiSIG_BPR_extra_filters is not None:
            eaiSIG_BPR_extra_filters.append(date_filter)
            eaiSIG_BPR_final_filters = " AND ".join(eaiSIG_BPR_extra_filters)
        else:
            eaiSIG_BPR_final_filters = date_filter

        eaiSIG_BPR_selected_fields = [
            "eaiSIG_BPR_CEXC_TYPE",
            "eaiSIG_BPR_CUST_S_UNID",
            "eaiSIG_BPR_CASE_S_ID",
        ]

        eaiSIG_BPR = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotron_eaiSIG_BPR",
            eaiSIG_BPR_final_filters,
            eaiSIG_BPR_selected_fields,
        )

        eaiSIG_BPR = eaiSIG_BPR.withColumn(
            "PartyRelationshipType",
            when(eaiSIG_BPR.eaiSIG_BPR_CEXC_TYPE == "05", "Primary Owner")
            .when(eaiSIG_BPR.eaiSIG_BPR_CEXC_TYPE == "23", "Guarantor")
            .otherwise("Co-Owner"),
        )

        # ================== eaiSIG_BP ==================

        eaiSIG_BP_extra_filters = None

        if eaiSIG_BP_extra_filters is not None:
            eaiSIG_BP_extra_filters.append(date_filter)
            eaiSIG_BP_final_filters = " AND ".join(eaiSIG_BP_extra_filters)
        else:
            eaiSIG_BP_final_filters = date_filter

        eaiSIG_BP_selected_fields = [
            "eaisig_bp_cust_s_unid",
            "eaiSIG_BP_CUST_VAT",
            "eaiSIG_BP_CUST_FNAME",
            "eaiSIG_BP_CUST_LNAME",
            "businessdate",
        ]

        eaiSIG_BP = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotron_eaiSIG_BP",
            eaiSIG_BP_final_filters,
            eaiSIG_BP_selected_fields,
        )

        eaiSIG_BP = eaiSIG_BP.withColumn(
            "PartyName",
            when(
                eaiSIG_BP.eaiSIG_BP_CUST_FNAME.isin(None),
                eaiSIG_BP.eaiSIG_BP_CUST_LNAME,
            ).otherwise(
                concat_ws(
                    " ", eaiSIG_BP.eaiSIG_BP_CUST_FNAME, eaiSIG_BP.eaiSIG_BP_CUST_LNAME
                )
            ),
        )

        # ============================================

        eaiSIG_BE = (
            eaiSIG_BE.withColumn(
                "ExposureStatusCode",
                when(
                    (eaiSIG_BE.eaiSIG_BE_CASE_STATUS1 == "A")
                    & (eaiSIG_BE.eaiSIG_BE_DEL_DAYS.cast("int") == 0),
                    1,
                )
                .when(
                    (eaiSIG_BE.eaiSIG_BE_CASE_STATUS1 == "A")
                    & (eaiSIG_BE.eaiSIG_BE_DEL_DAYS.cast("int") > 0),
                    2,
                )
                .when(eaiSIG_BE.eaiSIG_BE_CASE_STATUS1 == "J", 3)
                .when(eaiSIG_BE.eaiSIG_BE_CASE_STATUS1 == "L", 4),
            )
            .withColumn(
                "DueAmount",
                eaiSIG_BE.eaiSIG_BE_CASE_OVRDINT2.cast("decimal(26,2)")
                - eaiSIG_BE.eaiSIG_BE_CASE_OVRDINT1.cast("decimal(26,2)"),
            )
            .withColumn(
                "EverDenouncedFlag",
                when(eaiSIG_BE.eaiSIG_BE_CASE_STATUS1_DATE == "1753-01-01", False)
                .when(eaiSIG_BE.eaiSIG_BE_CASE_STATUS1_DATE.isNull(), False)
                .otherwise(True),
            )
            .withColumn(
                "EverDenouncedDate",
                when(eaiSIG_BE.eaiSIG_BE_CASE_STATUS1_DATE == "1753-01-01", lit(None))
                .when(eaiSIG_BE.eaiSIG_BE_CASE_STATUS1_DATE.isNull(), lit(None))
                .otherwise(eaiSIG_BE.eaiSIG_BE_CASE_STATUS1_DATE),
            )
            .withColumn(
                "ExposureStatus",
                when(
                    (eaiSIG_BE.eaiSIG_BE_CASE_STATUS1 == "A")
                    & (eaiSIG_BE.eaiSIG_BE_DEL_DAYS.cast("int") == 0),
                    "ΕΝΗΜΕΡΟ",
                )
                .when(
                    (eaiSIG_BE.eaiSIG_BE_CASE_STATUS1 == "A")
                    & (eaiSIG_BE.eaiSIG_BE_DEL_DAYS.cast("int") > 0),
                    "ΥΠΕΡΗΜΕΡΟ",
                )
                .when(eaiSIG_BE.eaiSIG_BE_CASE_STATUS1 == "J", "ΟΡΙΣΤΙΚΗ ΚΑΘΥΣΤΕΡΗΣΗ")
                .when(eaiSIG_BE.eaiSIG_BE_CASE_STATUS1 == "L", "ΕΠΙΣΦΑΛΕΙΑ"),
            )
            .withColumn(
                "InterestDuration",
                when(
                    eaiSIG_BE.eaiSIG_BE_CASE_INTERESTRATE_EXPDATE >= current_date(),
                    datediff(
                        current_date(), eaiSIG_BE.eaiSIG_BE_CASE_INTERESTRATE_EXPDATE
                    ),
                ).otherwise(lit(None)),
            )
            .withColumn(
                "SlaveFlag",
                when(
                    eaiSIG_BE.eaiSIG_BE_CASE_MASTER_ACC.cast("bigint") > 0,
                    True,
                ).otherwise(False),
            )
        )

        # ================== Joins ==================
        eaiSIG_BE_eaiSIG_BPR_join_condition = (
            eaiSIG_BE["eaiSIG_BE_CASE_S_ID"] == eaiSIG_BPR["eaiSIG_BPR_CASE_S_ID"]
        )

        eaiSIG_BPR_eaiSIG_BP_join_condition = (
            eaiSIG_BPR["eaisig_bpr_cust_s_unid"] == eaiSIG_BP["eaisig_bp_cust_s_unid"]
        )

        target_table_arotron = eaiSIG_BE.join(
            eaiSIG_BPR, eaiSIG_BE_eaiSIG_BPR_join_condition, "inner"
        ).join(eaiSIG_BP, eaiSIG_BPR_eaiSIG_BP_join_condition, "inner").join(
            portfolio,
            on=substring(eaiSIG_BE["eaiSIG_BE_CUST_S_UNID"], 1, 3) == portfolio["Prefix"],
            how="left"
        )

        target_table_arotron_ExposureId_primary_key_fields = [
            eaiSIG_BE.eaiSIG_BE_CASE_S_ID,
            eaiSIG_BPR.eaiSIG_BPR_CUST_S_UNID,
            eaiSIG_BPR.PartyRelationshipType,
            lit("arotron"),
        ]
        target_table_arotron_ExposureDebtorKey_primary_key_fields = [
            eaiSIG_BE.eaiSIG_BE_CASE_S_ID,
            eaiSIG_BPR.eaiSIG_BPR_CUST_S_UNID,
            eaiSIG_BPR.PartyRelationshipType,
        ]

        target_table_arotron = (
            target_table_arotron.withColumn(
                "ExposureId",
                sha2(
                    concat_ws(";", *target_table_arotron_ExposureId_primary_key_fields),
                    256,
                ),
            )
            .withColumn(
                "ExposureDebtorKey",
                concat_ws(
                    ";", *target_table_arotron_ExposureDebtorKey_primary_key_fields
                ),
            )
            .withColumn(
                "PrimaryPartyFlag",
                when(
                    eaiSIG_BE.eaiSIG_BE_CUST_S_UNID
                    == eaiSIG_BPR.eaiSIG_BPR_CUST_S_UNID,
                    True,
                ).otherwise(False),
            )
        )

        # ================== Selection of Final Columns ==================

        result_df_arotron = target_table_arotron.select(
            target_table_arotron.ExposureId.alias("ExposureId"),
            lit("arotron").alias("SourceSystem"),
            coalesce(portfolio.unit_code_profits, lit("-9998")).cast("string").alias("PortfolioKey"),
            coalesce(portfolio.PortfolioName, lit("Core Wholesale")).cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            target_table_arotron.ExposureDebtorKey.cast("string").alias(
                "ExposureDebtorKey"
            ),
            eaiSIG_BE.eaiSIG_BE_CASE_S_ID.cast("string").alias("ExposureKey"),
            lit(None).alias("CaseCode"),
            eaiSIG_BE.eaiSIG_BE_CASE_S_ID.cast("string").alias(
                "ExposureSourceSystemKey"
            ),
            eaiSIG_BE.eaiSIG_BE_CASE_CONTRNUM.cast("string").alias("AgreementKey"),
            eaiSIG_BE.eaiSIG_BE_CUST_S_UNID.cast("string").alias("PartyKey"),
            eaiSIG_BP.eaisig_bp_cust_s_unid.cast("string").alias("CDI"),
            eaiSIG_BP.PartyName.alias("PartyName"),
            eaiSIG_BP.eaiSIG_BP_CUST_VAT.cast("string").alias(
                "PartyTaxRegistrationNumber"
            ),
            eaiSIG_BPR.eaiSIG_BPR_CEXC_TYPE.cast("string").alias(
                "PartyRelationshipTypeCode"
            ),
            eaiSIG_BPR.PartyRelationshipType.cast("string").alias(
                "PartyRelationshipType"
            ),
            target_table_arotron.PrimaryPartyFlag.cast("boolean").alias(
                "PrimaryPartyFlag"
            ),
            eaiSIG_BE.eaiSIG_BE_SPRD_S_ID.cast("string").alias("ProductKey"),
            eaiSIG_BE.eaiSIG_BE_SPRD_S_ID.cast("int").alias("ProductCode"),
            lit(None).alias("ExposureTypeCode"),
            lit(None).alias("ExposureType"),
            eaiSIG_BE.ExposureStatusCode.cast("string").alias("ExposureStatusCode"),
            eaiSIG_BE.ExposureStatus.alias("ExposureStatus"),
            lit(None).alias("ExposureStatusDate"),
            eaiSIG_BE.eaiSIG_BE_CASE_OPNDATE.cast("date").alias("OpenDate"),
            lit(None).alias("ActiveExpiryDate"),
            eaiSIG_BE.eaiSIG_BE_CASE_EXPDATE.cast("date").alias("MaturityDate"),
            lit(None).alias("OriginBank"),
            eaiSIG_BE.eaiSIG_BE_CASE_SPENDLIM.cast("decimal(19,2)").alias(
                "OriginationAmount"
            ),
            eaiSIG_BE.eaiSIG_BE_CURR_CODE.alias("Currency"),
            lit(None).alias("BankingUnitCode"),
            lit(None).alias("BankingUnit"),
            eaiSIG_BE.eaiSIG_BE_CASE_SERVICE_TYPE.alias("MechanismCode"),
            eaiSIG_BE.eaiSIG_BE_CASE_SERVICE_TYPE.alias("Mechanism"),
            eaiSIG_BE.eaiSIG_BE_CASE_BRANCH.alias("BranchCode"),
            lit(None).alias("Branch"),
            lit(None).alias("PurposeReasonCode"),
            lit(None).alias("PurposeReason"),
            lit(None).alias("PurposeKTCode"),
            lit(None).alias("PurposeKT"),
            lit(None).alias("MaterialityStartDate"),
            lit(None).alias("DaysPastDueWithMateriality"),
            eaiSIG_BE.eaiSIG_BE_DEL_DAYS.cast("int").alias("DaysPastDue"),
            lit(None).alias("DaysPastDuePrincipal"),
            lit(None).alias("DaysPastDueInterest"),
            lit(None).alias("BucketCode"),
            lit(None).alias("Bucket"),
            lit(None).alias("FirstArrearDate"),
            lit(None).alias("ExpectedOverdueDate"),
            lit(None).alias("OverdueDate"),
            eaiSIG_BE.eaiSIG_BE_CASE_COLFRSDATE.cast("date").alias("DefaultDate"),
            eaiSIG_BE.eaiSIG_BE_CASE_STATUS1_DATE.cast("date").alias("DenouncedDate"),
            eaiSIG_BE.eaiSIG_BE_CASE_MASTER_ACC.alias("MasterExposureKey"),
            eaiSIG_BE.SlaveFlag.cast("boolean").alias("SlaveFlag"),
            eaiSIG_BE.eaiSIG_BE_CASE_PERIODCITY.cast("int").alias(
                "InstalmentFrequency"
            ),
            eaiSIG_BE.eaiSIG_BE_CASE_EMI_AMOUNT.cast("decimal(19,2)").alias(
                "InstallmentFixedAmount"
            ),
            eaiSIG_BE.eaiSIG_BE_CASE_DURATION.cast("int").alias("NumberOfInstallments"),
            lit(None).alias("FirstInstallmentDate"),
            lit(None).alias("NextInstallmentDate"),
            lit(None).alias("NextInstallmentAmount"),
            eaiSIG_BE.eaiSIG_BE_CASE_DUEDATE.cast("date").alias(
                "PreviousInstallmentDate"
            ),
            lit(None).alias("LastBillingDate"),
            lit(None).alias("NumberOfUnpaidBills"),
            lit(None).alias("LastRestructuringTypeCode"),
            lit(None).alias("LastRestructuringType"),
            eaiSIG_BE.eaiSIG_BE_CASE_STATUS4_DATE.cast("date").alias(
                "LastRestructuringDate"
            ),
            lit(None).alias("RestructuringPEEDate"),
            lit(None).alias("RestructuringPEETypeCode"),
            lit(None).alias("RestructuringPEEType"),
            lit(None).alias("ObjectiveStatusCode"),
            lit(None).alias("ObjectiveStatus"),
            lit(None).alias("ObjectiveStatusDate"),
            eaiSIG_BE.eaiSIG_BE_CASE_GRCP_EXPDATE.cast("date").alias(
                "GracePeriodEndDate"
            ),
            lit(None).alias("GracePeriodTypeCode"),
            lit(None).alias("GracePeriodType"),
            lit(None).alias("GracePeriodInterestPayableFlag"),
            lit(None).alias("GracePeriodNextInterestDate"),
            lit(None).alias("GracePeriodInterestFrequency"),
            eaiSIG_BE.eaiSIG_BE_CASE_DEBT.cast("decimal(19,2)").alias("OffBalance"),
            eaiSIG_BE.eaiSIG_BE_CASE_OUTBAL.cast("decimal(19,2)").alias("OnBalance"),
            eaiSIG_BE.eaiSIG_BE_CASE_OUTBAL.cast("string").alias(
                "OutstandingTotalBalance"
            ),
            eaiSIG_BE.eaiSIG_BE_CASE_CAPAMT.cast("decimal(19,2)").alias("Principal"),
            eaiSIG_BE.eaiSIG_BE_CASE_REGINT2.cast("decimal(19,2)").alias("Interest"),
            eaiSIG_BE.eaiSIG_BE_CASE_FRZAMT.cast("decimal(19,2)").alias(
                "UnrealizedInterest"
            ),
            lit(None).alias("Commission"),
            eaiSIG_BE.eaiSIG_BE_CASE_OTHERAMT1.cast("decimal(19,2)").alias("Expense"),
            lit(None).alias("Accrued"),
            eaiSIG_BE.DueAmount.cast("decimal(26,4)").alias("DueAmount"),
            eaiSIG_BE.eaiSIG_BE_CASE_OVRDINT2.cast("decimal(19,2)").alias(
                "DuePrincipal"
            ),
            eaiSIG_BE.eaiSIG_BE_CASE_OVRDINT1.cast("decimal(19,2)").alias(
                "DueInterest"
            ),
            eaiSIG_BE.eaiSIG_BE_CASE_WOFFAMT_INTEREST.cast("decimal(19,2)").alias(
                "DueUnrealizedInterest"
            ),
            lit(None).alias("DueCommission"),
            lit(None).alias("DueExpense"),
            lit(None).alias("DueAccrued"),
            lit(None).alias("PenaltyInterest"),
            lit(None).alias("PenaltyUnrealizedInterest"),
            eaiSIG_BE.eaiSIG_BE_CASE_INT_RATE.cast("decimal(19,6)").alias(
                "InterestRate"
            ),
            eaiSIG_BE.eaiSIG_BE_CASE_INTEREST_RATE_CATEGORY.cast("int").alias(
                "InterestCode"
            ),
            lit(None).alias("InterestDescription"),
            eaiSIG_BE.eaiSIG_BE_INTEREST_RATE_BASE.cast("decimal(19,6)").alias(
                "BaseRate"
            ),
            lit(None).alias("BaseCode"),
            lit(None).alias("BaseDescription"),
            eaiSIG_BE.eaiSIG_BE_CASE_PRCT_CONTRB.cast("decimal(19,6)").alias(
                "N128Rate"
            ),
            eaiSIG_BE.eaiSIG_BE_SPREAD_INTEREST_MARGIN.cast("decimal(19,6)").alias(
                "SpreadRate"
            ),
            eaiSIG_BE.eaiSIG_BE_CASE_INTERESTRATE_DEFAULT.cast("decimal(19,6)").alias(
                "PenaltyRate"
            ),
            eaiSIG_BE.eaiSIG_BE_CASE_RATE_CAT.alias("InterestRateTypeCode"),
            lit(None).alias("InterestRateType"),
            lit(None).alias("SpreadCurrencyCode"),
            lit(None).alias("SpreadCurrency"),
            lit(None).alias("SpreadPeriod"),
            eaiSIG_BE.InterestDuration.alias("InterestDuration"),
            lit(None).alias("DayCounts"),
            lit(None).alias("InterestFrequency"),
            lit(None).alias("InterestFixedDate"),
            lit(None).alias("InterestValidityDate"),
            lit(None).alias("InterestChangeStartDate"),
            lit(None).alias("InterestChangeEndDate"),
            lit(None).alias("InterestRateUpdateDate"),
            lit(None).alias("InterestCalcStopFlag"),
            lit(None).alias("AutoRenewalFlag"),
            eaiSIG_BE.eaiSIG_BE_CASE_ISOPEN.cast("boolean").alias("IsActiveFlag"),
            lit(None).alias("WriteOffAmount"),
            lit(None).alias("WriteOffDate"),
            eaiSIG_BE.eaiSIG_BE_CASE_BUSUNIT.cast("string").alias("AssetSegmentCode"),
            eaiSIG_BE.eaiSIG_BE_CASE_BUSUNIT.cast("string").alias("AssetSegment"),
            lit(None).cast("string").alias("DiasNumber"),
            lit(None).cast("string").alias("ProvisionFrozenInterestAmount"),
            lit(None).cast("string").alias("AccountNticeBUKey"),
            lit(None).cast("string").alias("SayFlag"),
            lit(None).cast("string").alias("InsuranceCompany"),
            lit(None).cast("string").alias("PhysicalNumber"),
            lit(None).cast("string").alias("PreCourtDate"),
            lit(None).cast("string").alias("CourtDate"),
            lit(None).cast("string").alias("ContractSigningDate"),
            lit(None).cast("string").alias("ExposureLimitAmount"),
            lit(None).cast("decimal(15,2)").alias("BaloonAmount"),
            lit(None).cast("integer").alias("AdministrationStatusCode"),
            lit(None).cast("integer").alias("AdministrationStatus"),
            lit(None).cast("string").alias("tenor"),
            lit(None).cast("boolean").alias("IsSoldFlag"),
            lit(None).cast("string").alias("IsSubsidyFlag"),
            lit(None).cast("boolean").alias("IsGuaranteeFlag"),
            eaiSIG_BE.EverDenouncedFlag.cast("string").alias("EverDenouncedFlag"),
            eaiSIG_BE.EverDenouncedDate.cast("string").alias("EverDenouncedDate"),
            eaiSIG_BE.eaiSIG_BE_CASE_SPV.cast("string").alias("SPVCode"),
            eaiSIG_BE.eaiSIG_BE_CASE_SPV.cast("string").alias("SPV"),
            lit(None).alias("TotalOverdueInsterest"),
            lit(None).alias("TotalOverdueInstallments"),
            lit(None).cast("decimal(19,4)").alias("OutstandingTotalBalanceMLT"),
            lit(None).alias("AllBorrowersDeceasedFlag"),
            lit(None).alias("TreatmentBranchCode"),
            lit(None).alias("TreatmentBranch"),
            lit(None).alias("LetterStatusCode"),
            lit(None).alias("LetterStatus"),
            lit(None).alias("LetterStatusDate"),
            lit(None).alias("CoCStatusCode"),
            lit(None).alias("CoCStatus"),
            lit(None).alias("CoCAccountStatusDate"),
            lit(None).alias("FirstLetterDate"),
            lit(None).alias("SecondLetterDate"),
            lit(None).alias("ThirdLetterDate"),
            lit(None).alias("FirstLetterReminderDate"),
            lit(None).alias("FirstLetterFlag"),
            lit(None).alias("FirstLetterReminderFlag"),
            lit(None).alias("SecondLetterFlag"),
            lit(None).alias("ThirdLetterFlag"),
            lit(None).alias("ExclusionDate"),
            lit(None).alias("ExclusionReasonCode"),
            lit(None).alias("ExclusionReason"),
            lit(None).alias("LegalActionPriority"),
            lit(None).alias("DueDebtAmount"),
            lit(None).alias("JustificationCode"),
            lit(None).alias("JustificationComments"), 
            eaiSIG_BE.businessdate.alias("BusinessDate"),
            lit(None).cast("Boolean").alias("ForcedOnboarding"),
            lit(None).cast("Boolean").alias("HolisticFlag"),
            lit(None).cast("String").alias("ForcedOnboardReason"), 
            lit(None).cast("decimal(15,2)").alias("url_nrm_int_amn"),
        )

        # ========================== Append ==========================

        result_df_arotron.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)
    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_exposure_arotron(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)